In [2]:
import findspark
findspark.init()

In [3]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
spark=SparkSession.builder.getOrCreate()
spark

24/11/14 23:16:20 WARN Utils: Your hostname, Nishants-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
24/11/14 23:16:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/14 23:16:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 55481)
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.12/3.12.4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.12/3.12.4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/homebrew/Cellar/python@3.12/3.12.4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/Users/nishantketu/Projects/python-playground/venv/lib/py

In [25]:
# df=spark.read.csv(['people-100.csv','people-100-2.csv'],header=True,inferSchema=True)
df=spark.read.csv('./csvfile/',header=True,inferSchema=True)
df.printSchema()
df=df.limit(5)
df.show(100)

root
 |-- Index: string (nullable = true)
 |-- User Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Date of birth: date (nullable = true)
 |-- Job Title: string (nullable = true)

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+------------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|         Job Title|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+------------------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|   Games developer|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24|    Phytotherapist|
|    3|

In [11]:
# where sex is male add male or female

from pyspark.sql.functions import when
df.withColumn("gender",when(df.Sex=="Male",'M').when(df.Sex=="Female",'F').otherwise("F")).show()

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+------------------+------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|         Job Title|gender|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+------------------+------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|   Games developer|     M|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24|    Phytotherapist|     F|
|    3|DbeAb8CcdfeFC2c|  Kristine|   Travis|  Male|bthompson@example...|        277.609.7938|   1992-07-02|         Homeopath|     M|
|    4|A31Bee3c201ef58|   Yesenia| Martinez|  Male|kaitlinkaiser@exa...|        584.094.6111|   2017-08-03| Market researcher|     M|
|    5|1bA7A3dc874da3c|      Lori|     Todd|  Male|buchananman

In [29]:
# replace a letter from name

from pyspark.sql.functions import regexp_replace,col
df=df.withColumn("new",regexp_replace(col("First Name"),"S","X"))
df.show()
df.drop(col('First Name')).withColumnRenamed('new','name').show()

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+------------------+--------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|         Job Title|     new|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+------------------+--------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|   Games developer|  Xhelby|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24|    Phytotherapist| Phillip|
|    3|DbeAb8CcdfeFC2c|  Kristine|   Travis|  Male|bthompson@example...|        277.609.7938|   1992-07-02|         Homeopath|Kristine|
|    4|A31Bee3c201ef58|   Yesenia| Martinez|  Male|kaitlinkaiser@exa...|        584.094.6111|   2017-08-03| Market researcher| Yesenia|
|    5|1bA7A3dc874da3c|      Lori|     Todd|  Ma

In [33]:
# add current date and current time stamp
from pyspark.sql.functions import current_date,current_timestamp,date_format

df.withColumn("current date",current_date()).withColumn("current time",current_timestamp()).show(truncate=False)

# only taking year from dob
df.select(date_format(col('Date of birth'),"yyyy")).show(truncate=False)

+-----+---------------+----------+---------+------+--------------------------+----------------------+-------------+------------------+--------+------------+--------------------------+
|Index|User Id        |First Name|Last Name|Sex   |Email                     |Phone                 |Date of birth|Job Title         |new     |current date|current time              |
+-----+---------------+----------+---------+------+--------------------------+----------------------+-------------+------------------+--------+------------+--------------------------+
|1    |88F7B33d2bcf9f5|Shelby    |Terrell  |Male  |elijah57@example.net      |001-084-906-7849x73518|1945-10-26   |Games developer   |Xhelby  |2024-11-14  |2024-11-14 23:49:20.469144|
|2    |f90cD3E76f1A9b9|Phillip   |Summers  |Female|bethany14@example.com     |214.112.6044x4913     |1910-03-24   |Phytotherapist    |Phillip |2024-11-14  |2024-11-14 23:49:20.469144|
|3    |DbeAb8CcdfeFC2c|Kristine  |Travis   |Male  |bthompson@example.com     |27

In [34]:
data=[("nishant",353,'python','M',78),("ketu",358,'sql','F',98),("meh",153,'python','M',65),("le",353,'sql','M',67),("be",458,'sql','F',24)]
df=spark.createDataFrame(data,['name','salary','skill','gender','bonus'])

In [39]:
from pyspark.sql.functions import count

df.groupBy('skill').agg(count('*').alias("count")).show()

+------+-----+
| skill|count|
+------+-----+
|python|    2|
|   sql|    3|
+------+-----+



In [42]:
# select skill, sum of salary,sort by skill
from pyspark.sql.functions import count,sum

df.groupBy('skill').agg(sum('salary').alias('skill_salary')).orderBy('skill').show()

+------+------------+
| skill|skill_salary|
+------+------------+
|python|         506|
|   sql|        1169|
+------+------------+



In [48]:
# having avg salary>300
from pyspark.sql.functions import count,sum,avg

df.groupBy('skill').agg(avg('salary').alias('avg_salary')).where('avg_salary>300').show()

+-----+-----------------+
|skill|       avg_salary|
+-----+-----------------+
|  sql|389.6666666666667|
+-----+-----------------+

